In [1]:
import torch
print("CUDA 사용 가능:", torch.cuda.is_available())
print("GPU 이름:", torch.cuda.get_device_name(0))


CUDA 사용 가능: True
GPU 이름: NVIDIA GeForce RTX 3060


In [1]:
import numpy as np

for i in range(10):
    lotto = np.random.choice(np.arange(1, 46), 7, replace=False)
    main, bonus = np.sort(lotto[:6]), lotto[6]
    print(f"{i+1:2}회차 → 당첨 번호: {main} / 보너스: {bonus}")

 1회차 → 당첨 번호: [ 1  9 12 15 27 32] / 보너스: 20
 2회차 → 당첨 번호: [ 1  4  6  7 16 31] / 보너스: 24
 3회차 → 당첨 번호: [15 22 27 31 32 41] / 보너스: 7
 4회차 → 당첨 번호: [ 5  7 17 21 26 30] / 보너스: 37
 5회차 → 당첨 번호: [20 25 26 39 44 45] / 보너스: 18
 6회차 → 당첨 번호: [ 7 16 17 23 28 37] / 보너스: 11
 7회차 → 당첨 번호: [20 22 26 28 35 41] / 보너스: 18
 8회차 → 당첨 번호: [11 14 32 36 44 45] / 보너스: 20
 9회차 → 당첨 번호: [14 17 18 32 42 43] / 보너스: 15
10회차 → 당첨 번호: [ 6 11 14 31 33 44] / 보너스: 30


# make csv 1

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import sys

def get_lotto_numbers(draw_no):
    try:
        url = f"https://www.dhlottery.co.kr/gameResult.do?method=byWin&drwNo={draw_no}"
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
        }
        res = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")

        win_nums = soup.select(".win_result .nums span.ball_645")
        bonus_num = soup.select_one(".win_result .bonus span.ball_645")

        if len(win_nums) < 6 or bonus_num is None:
            print(f"[{draw_no}회] 데이터 부족. 건너뜀.")
            return None

        win_nums = [int(num.text) for num in win_nums]
        bonus = int(bonus_num.text)

        return {
            "회차": draw_no,
            "번호1": win_nums[0],
            "번호2": win_nums[1],
            "번호3": win_nums[2],
            "번호4": win_nums[3],
            "번호5": win_nums[4],
            "번호6": win_nums[5],
            "보너스": bonus
        }

    except Exception as e:
        print(f"[{draw_no}회] 오류 발생: {e}")
        return "error"

all_data = []
fail_list = []

total = 1174

for i in range(1, total + 1):
    result = get_lotto_numbers(i)
    if result == "error":
        fail_list.append(i)
    elif result:
        all_data.append(result)

    progress = (i / total) * 100
    sys.stdout.write(f"\r진행률: {progress:.1f}% ({i} / {total})")
    sys.stdout.flush()

    time.sleep(0.5)

print("\n⏳ 1차 완료. 오류 회차 수: {}개. 재시도 중...".format(len(fail_list)))

retry_list = []
for i in fail_list:
    time.sleep(1)
    result = get_lotto_numbers(i)
    if result == "error":
        retry_list.append(i)
    elif result:
        all_data.append(result)

print(f" 최종 완료! 실패한 회차 수: {len(retry_list)}개")

df = pd.DataFrame(all_data)
df = df.sort_values("회차")
df.to_csv("lotto_numbers.csv", index=False, encoding='utf-8-sig')
print("💾 로또 데이터 저장 완료!")

if retry_list:
    print("다시 시도해도 실패한 회차:", retry_list)

# make scv2

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import sys
import os

def get_latest_draw_no():
    url = "https://www.dhlottery.co.kr/gameResult.do?method=byWin"
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    res = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(res.text, "html.parser")
    # 최신 회차 번호가 들어있는 태그 선택
    latest_no_tag = soup.select_one(".nums > strong")
    if latest_no_tag:
        return int(latest_no_tag.text.strip())
    else:
        return None

def get_lotto_numbers(draw_no):
    try:
        url = f"https://www.dhlottery.co.kr/gameResult.do?method=byWin&drwNo={draw_no}"
        headers = {
            "User-Agent": "Mozilla/5.0"
        }
        res = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")

        win_nums = soup.select(".win_result .nums span.ball_645")
        bonus_num = soup.select_one(".win_result .bonus span.ball_645")

        if len(win_nums) < 6 or bonus_num is None:
            print(f"[{draw_no}회] 데이터 부족. 건너뜀.")
            return None

        win_nums = [int(num.text) for num in win_nums]
        bonus = int(bonus_num.text)

        return {
            "회차": draw_no,
            "번호1": win_nums[0],
            "번호2": win_nums[1],
            "번호3": win_nums[2],
            "번호4": win_nums[3],
            "번호5": win_nums[4],
            "번호6": win_nums[5],
            "보너스": bonus
        }

    except Exception as e:
        print(f"[{draw_no}회] 오류 발생: {e}")
        return "error"

# 저장된 데이터가 있으면 불러오기
csv_path = "lotto_numbers.csv"
if os.path.exists(csv_path):
    df_existing = pd.read_csv(csv_path)
    start_no = df_existing["회차"].max() + 1
else:
    df_existing = pd.DataFrame()
    start_no = 1

# 최신 회차 가져오기
latest_no = get_latest_draw_no()
if latest_no is None:
    print("최신 회차를 가져오지 못했습니다. 수동으로 total 지정 필요.")
    total = 1174  # 수동 지정 가능
else:
    total = latest_no

print(f"크롤링 시작: {start_no}회부터 {total}회까지")

all_data = []

fail_list = []

for i in range(start_no, total + 1):
    result = get_lotto_numbers(i)
    if result == "error":
        fail_list.append(i)
    elif result:
        all_data.append(result)

    progress = (i - start_no + 1) / (total - start_no + 1) * 100
    sys.stdout.write(f"\r진행률: {progress:.1f}% ({i} / {total})")
    sys.stdout.flush()

    time.sleep(0.5)

print("\n⏳ 1차 완료. 오류 회차 수: {}개. 재시도 중...".format(len(fail_list)))

retry_list = []
for i in fail_list:
    time.sleep(1)
    result = get_lotto_numbers(i)
    if result == "error":
        retry_list.append(i)
    elif result:
        all_data.append(result)

print(f" 최종 완료! 실패한 회차 수: {len(retry_list)}개")

# 기존 데이터랑 합치기
if not df_existing.empty:
    df_new = pd.DataFrame(all_data)
    df_all = pd.concat([df_existing, df_new], ignore_index=True)
    df_all = df_all.drop_duplicates(subset="회차").sort_values("회차")
else:
    df_all = pd.DataFrame(all_data)

df_all.to_csv(csv_path, index=False, encoding='utf-8-sig')
print("💾 로또 데이터 저장 완료!")

if retry_list:
    print("다시 시도해도 실패한 회차:", retry_list)


최신 회차를 가져오지 못했습니다. 수동으로 total 지정 필요.
크롤링 시작: 1175회부터 1174회까지

⏳ 1차 완료. 오류 회차 수: 0개. 재시도 중...
 최종 완료! 실패한 회차 수: 0개
💾 로또 데이터 저장 완료!


## 1. 데이터 전처리

In [ ]:
import pandas as pd
import numpy as np

# 데이터 불러오기
df = pd.read_csv("lotto_numbers.csv")

# 번호들 one-hot encoding 함수 (1~45 숫자 각각 0/1로 표시)
def one_hot(row):
    arr = np.zeros(45, dtype=int)
    numbers = row[['번호1','번호2','번호3','번호4','번호5','번호6']].values
    for n in numbers:
        arr[n-1] = 1
    return arr

X = []
Y = []

# 한 회차의 번호로 다음 회차 번호 예측하도록 데이터 생성
for i in range(len(df)-1):
    X.append(one_hot(df.iloc[i]))
    Y.append(one_hot(df.iloc[i+1]))

X = np.array(X)
Y = np.array(Y)

print("X shape:", X.shape)  # (데이터수, 45)
print("Y shape:", Y.shape)


## 2. 모델 만들기 (keras) & 학습

In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import numpy as np

# X, Y는 미리 준비된 상태라고 가정
# X.shape -> (샘플 수, 특성 수)
# Y.shape -> (샘플 수, 45) 멀티라벨 (0 또는 1)

# 1) 데이터 분포 확인
print("레이블별 합 (각 번호 출현 횟수):", np.sum(Y, axis=0))
print("샘플별 라벨 개수 평균:", np.mean(np.sum(Y, axis=1)))

# 2) 훈련/검증 분리
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

# 3) 모델 정의
model = Sequential([
    Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(45, activation='sigmoid')  # 45개 번호 각각 확률 출력
])

# 4) 모델 컴파일 (멀티라벨 분류에 적합한 binary_crossentropy 사용)
model.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.001)
)

# 5) EarlyStopping 설정
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# 6) 모델 학습
history = model.fit(
    X_train, Y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_val, Y_val),
    callbacks=[early_stop],
    verbose=1
)

# 7) 검증 데이터 예측
y_pred_prob = model.predict(X_val)

# 8) 임계값 기준 이진화 (기본 0.5, 필요시 조정 가능)
threshold = 0.5
y_pred = (y_pred_prob > threshold).astype(int)

# 9) 평가 지표 출력
print("\n✅ F1 Score (macro):", f1_score(Y_val, y_pred, average='macro'))
print("✅ F1 Score (micro):", f1_score(Y_val, y_pred, average='micro'))

print("\n📊 Classification Report:")
print(classification_report(Y_val, y_pred, zero_division=0))


레이블별 합 (각 번호 출현 횟수): [160 149 161 153 147 156 162 149 127 154 160 172 170 167 157 156 164 169
 160 163 161 138 138 157 145 158 168 145 146 150 158 137 166 180 155 156
 163 162 161 164 141 150 159 156 168]
샘플별 라벨 개수 평균: 6.0
Epoch 1/100
30/30 [==============================] - 1s 8ms/step - loss: 0.5429 - val_loss: 0.4126
Epoch 2/100
30/30 [==============================] - 0s 4ms/step - loss: 0.4114 - val_loss: 0.3951
Epoch 3/100
30/30 [==============================] - 0s 5ms/step - loss: 0.4029 - val_loss: 0.3943
Epoch 4/100
30/30 [==============================] - 0s 5ms/step - loss: 0.3987 - val_loss: 0.3948
Epoch 5/100
30/30 [==============================] - 0s 4ms/step - loss: 0.3968 - val_loss: 0.3945
Epoch 6/100
30/30 [==============================] - 0s 4ms/step - loss: 0.3950 - val_loss: 0.3944
Epoch 7/100
30/30 [==============================] - 0s 5ms/step - loss: 0.3931 - val_loss: 0.3947
Epoch 8/100
30/30 [==============================] - 0s 4ms/step - loss: 0.3923 - va

# 계산기

In [33]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input, Concatenate, LSTM, Reshape
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# 재현 가능한 결과를 위한 시드 설정
np.random.seed(42)
tf.random.set_seed(42)

class LottoPredictionModel:
    def __init__(self, csv_path='lotto_numbers.csv'):
        self.csv_path = csv_path
        self.model = None
        self.scaler = StandardScaler()
        self.window_size = 10  # 더 긴 시퀀스로 패턴 학습
        
    def load_and_preprocess_data(self):
        """데이터 로드 및 전처리"""
        print("📊 데이터 로딩 중...")
        self.df = pd.read_csv(self.csv_path)
        
        # 회차 기준으로 정렬 (최신 데이터가 마지막에 오도록)
        if '회차' in self.df.columns:
            self.df = self.df.sort_values('회차').reset_index(drop=True)
        
        print(f"✅ 총 {len(self.df)}회차 데이터 로드 완료")
        
    def numbers_to_onehot(self, numbers, size=45):
        """번호를 원-핫 인코딩"""
        onehot = np.zeros(size)
        for n in numbers:
            if 1 <= n <= 45:
                onehot[n-1] = 1
        return onehot
    
    def create_statistical_features(self, window_data):
        """통계적 특성 추출"""
        features = []
        
        # 각 회차별 번호들
        all_numbers = []
        for _, row in window_data.iterrows():
            nums = row[['번호1','번호2','번호3','번호4','번호5','번호6']].values
            all_numbers.extend(nums)
        
        # 번호별 출현 빈도
        counter = Counter(all_numbers)
        freq_features = [counter.get(i, 0) for i in range(1, 46)]
        
        # 연속번호 패턴
        consecutive_count = 0
        for _, row in window_data.iterrows():
            nums = sorted(row[['번호1','번호2','번호3','번호4','번호5','번호6']].values)
            for i in range(len(nums)-1):
                if nums[i+1] - nums[i] == 1:
                    consecutive_count += 1
        
        # 홀짝 비율
        odd_count = sum(1 for n in all_numbers if n % 2 == 1)
        odd_ratio = odd_count / len(all_numbers)
        
        # 구간별 분포 (1-15, 16-30, 31-45)
        zone1 = sum(1 for n in all_numbers if 1 <= n <= 15)
        zone2 = sum(1 for n in all_numbers if 16 <= n <= 30)
        zone3 = sum(1 for n in all_numbers if 31 <= n <= 45)
        total = len(all_numbers)
        zone_ratios = [zone1/total, zone2/total, zone3/total]
        
        # 평균, 분산
        avg_num = np.mean(all_numbers)
        var_num = np.var(all_numbers)
        
        features.extend(freq_features)  # 45개
        features.extend([consecutive_count, odd_ratio, avg_num, var_num])  # 4개
        features.extend(zone_ratios)  # 3개
        
        return np.array(features)
    
    def create_dataset(self):
        """학습용 데이터셋 생성"""
        print("🔄 데이터셋 생성 중...")
        
        X_onehot_list = []
        X_stat_list = []
        Y_list = []
        
        for i in range(len(self.df) - self.window_size):
            # 원-핫 인코딩 특성
            onehot_vector = []
            window_data = self.df.iloc[i:i+self.window_size]
            
            for j in range(len(window_data)):
                nums = window_data.iloc[j][['번호1','번호2','번호3','번호4','번호5','번호6','보너스']].values
                onehot_vector.append(self.numbers_to_onehot(nums))
            
            X_onehot_list.append(np.array(onehot_vector))  # (window_size, 45)
            
            # 통계적 특성
            stat_features = self.create_statistical_features(window_data)
            X_stat_list.append(stat_features)
            
            # 타겟 (다음 회차)
            next_nums = self.df.iloc[i + self.window_size][['번호1','번호2','번호3','번호4','번호5','번호6','보너스']].values
            Y_list.append(self.numbers_to_onehot(next_nums))
        
        self.X_onehot = np.array(X_onehot_list)
        self.X_stat = np.array(X_stat_list)
        self.Y = np.array(Y_list)
        
        # 통계 특성 정규화
        self.X_stat = self.scaler.fit_transform(self.X_stat)
        
        print(f"✅ 데이터셋 생성 완료: {len(self.X_onehot)}개 샘플")
        print(f"   - 원핫 특성: {self.X_onehot.shape}")
        print(f"   - 통계 특성: {self.X_stat.shape}")
        print(f"   - 타겟: {self.Y.shape}")
    
    def build_model(self):
        """하이브리드 딥러닝 모델 구축"""
        print("🏗️ 모델 구축 중...")
        
        # 원-핫 인코딩 입력 (시퀀스 데이터)
        onehot_input = Input(shape=(self.window_size, 45), name='onehot_input')
        
        # LSTM으로 시퀀스 패턴 학습
        lstm_out = LSTM(128, return_sequences=True, dropout=0.3)(onehot_input)
        lstm_out = LSTM(64, dropout=0.3)(lstm_out)
        lstm_out = Dense(32, activation='relu')(lstm_out)
        
        # 통계적 특성 입력
        stat_input = Input(shape=(self.X_stat.shape[1],), name='stat_input')
        stat_dense = Dense(64, activation='relu')(stat_input)
        stat_dense = Dropout(0.3)(stat_dense)
        stat_dense = Dense(32, activation='relu')(stat_dense)
        
        # 두 경로 결합
        combined = Concatenate()([lstm_out, stat_dense])
        combined = BatchNormalization()(combined)
        combined = Dropout(0.4)(combined)
        
        # 최종 예측 레이어
        x = Dense(256, activation='relu')(combined)
        x = BatchNormalization()(x)
        x = Dropout(0.4)(x)
        
        x = Dense(128, activation='relu')(x)
        x = BatchNormalization()(x)
        x = Dropout(0.3)(x)
        
        x = Dense(64, activation='relu')(x)
        x = Dropout(0.2)(x)
        
        # 출력층 - 각 번호별 확률
        output = Dense(45, activation='sigmoid', name='output')(x)
        
        self.model = Model(inputs=[onehot_input, stat_input], outputs=output)
        
        # 컴파일
        self.model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        
        print("✅ 모델 구축 완료")
        self.model.summary()
    
    def train_model(self):
        """모델 학습"""
        print("🎯 모델 학습 시작...")
        
        # 데이터 분할
        indices = np.arange(len(self.X_onehot))
        train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)
        
        X_onehot_train, X_onehot_val = self.X_onehot[train_idx], self.X_onehot[val_idx]
        X_stat_train, X_stat_val = self.X_stat[train_idx], self.X_stat[val_idx]
        Y_train, Y_val = self.Y[train_idx], self.Y[val_idx]
        
        # 콜백 설정
        callbacks = [
            EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6)
        ]
        
        # 학습
        history = self.model.fit(
            [X_onehot_train, X_stat_train], Y_train,
            epochs=100,
            batch_size=16,
            validation_data=([X_onehot_val, X_stat_val], Y_val),
            callbacks=callbacks,
            verbose=1
        )
        
        # 검증 데이터로 평가
        self.evaluate_model(X_onehot_val, X_stat_val, Y_val)
        
        return history
    
    def evaluate_model(self, X_onehot_val, X_stat_val, Y_val):
        """모델 평가"""
        print("\n📈 모델 평가 중...")
        
        y_pred_prob = self.model.predict([X_onehot_val, X_stat_val], verbose=0)
        
        # 임계값 최적화
        best_f1 = 0
        best_threshold = 0.5
        
        for threshold in np.arange(0.3, 0.8, 0.05):
            y_pred_bin = (y_pred_prob > threshold).astype(int)
            f1 = f1_score(Y_val, y_pred_bin, average='macro', zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_threshold = threshold
        
        print(f"✅ 최적 임계값: {best_threshold:.2f}")
        print(f"✅ 최고 F1 Score: {best_f1:.4f}")
        
        # 최적 임계값으로 최종 평가
        y_pred_bin = (y_pred_prob > best_threshold).astype(int)
        
        print(f"✅ F1 Score (macro): {f1_score(Y_val, y_pred_bin, average='macro', zero_division=0):.4f}")
        print(f"✅ F1 Score (micro): {f1_score(Y_val, y_pred_bin, average='micro', zero_division=0):.4f}")
        
        # 완전 일치 정확도
        accuracy = np.mean(np.all(y_pred_bin == Y_val, axis=1))
        print(f"✅ 전체 번호 완전 일치율: {accuracy:.6f}")
        
        self.best_threshold = best_threshold
    
    def predict_next_numbers(self, num_predictions=10):
        """다음 회차 번호 예측"""
        print(f"\n🎯 다음 회차 로또 번호 {num_predictions}개 조합 예측...")
        
        # 최근 window_size개 회차 데이터 준비
        recent_data = self.df.tail(self.window_size)
        
        # 원-핫 인코딩
        onehot_vector = []
        for i in range(len(recent_data)):
            nums = recent_data.iloc[i][['번호1','번호2','번호3','번호4','번호5','번호6','보너스']].values
            onehot_vector.append(self.numbers_to_onehot(nums))
        
        X_onehot_pred = np.array([onehot_vector])
        
        # 통계적 특성
        stat_features = self.create_statistical_features(recent_data)
        X_stat_pred = self.scaler.transform([stat_features])
        
        # 예측
        predictions = []
        for i in range(num_predictions):
            # 약간의 노이즈 추가로 다양성 확보
            noise_factor = 0.01
            X_onehot_noisy = X_onehot_pred + np.random.normal(0, noise_factor, X_onehot_pred.shape)
            X_stat_noisy = X_stat_pred + np.random.normal(0, noise_factor, X_stat_pred.shape)
            
            y_pred_prob = self.model.predict([X_onehot_noisy, X_stat_noisy], verbose=0)[0]
            
            # 상위 확률 번호들 선택 (약간의 랜덤성 추가)
            top_indices = np.argsort(y_pred_prob)[-15:]  # 상위 15개 후보
            
            # 가중 랜덤 선택으로 7개 선택
            probs = y_pred_prob[top_indices]
            probs = probs / np.sum(probs)  # 정규화
            
            selected_indices = np.random.choice(top_indices, size=7, replace=False, p=probs)
            selected_numbers = selected_indices + 1
            
            # 메인 6개, 보너스 1개로 분리
            np.random.shuffle(selected_numbers)
            main_numbers = sorted(selected_numbers[:6])
            bonus_number = selected_numbers[6]
            
            predictions.append({
                'main': main_numbers,
                'bonus': bonus_number,
                'confidence': np.mean(y_pred_prob[selected_indices])
            })
        
        # 신뢰도 순으로 정렬
        predictions.sort(key=lambda x: x['confidence'], reverse=True)
        
        print("\n🎲 예측 결과:")
        print("=" * 60)
        for i, pred in enumerate(predictions, 1):
            main_str = ', '.join(map(str, pred['main']))
            print(f"조합 {i:2d} │ [{main_str}] + {pred['bonus']} │ 신뢰도: {pred['confidence']:.3f}")
        
        return predictions
    
    def run_full_pipeline(self):
        """전체 파이프라인 실행"""
        print("🚀 로또 번호 예측 시스템 시작!")
        print("=" * 50)
        
        self.load_and_preprocess_data()
        self.create_dataset()
        self.build_model()
        history = self.train_model()
        predictions = self.predict_next_numbers(10)
        
        print("\n🎉 예측 완료! 행운을 빕니다! 🍀")
        return predictions

# 실행
if __name__ == "__main__":
    # 모델 인스턴스 생성 및 실행
    lotto_model = LottoPredictionModel('lotto_numbers.csv')
    predictions = lotto_model.run_full_pipeline()
    
    print("\n💡 추가 팁:")
    print("- 로또는 완전 랜덤이므로 어떤 모델도 100% 예측할 수 없습니다")
    print("- 이 모델은 과거 패턴을 학습하여 통계적으로 가능성을 제시합니다")
    print("- 책임감 있는 복권 구매를 권장합니다")

🚀 로또 번호 예측 시스템 시작!
📊 데이터 로딩 중...
✅ 총 1174회차 데이터 로드 완료
🔄 데이터셋 생성 중...
✅ 데이터셋 생성 완료: 1164개 샘플
   - 원핫 특성: (1164, 10, 45)
   - 통계 특성: (1164, 52)
   - 타겟: (1164, 45)
🏗️ 모델 구축 중...
✅ 모델 구축 완료
Model: "model_3"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 onehot_input (InputLayer)      [(None, 10, 45)]     0           []                               
                                                                                                  
 stat_input (InputLayer)        [(None, 52)]         0           []                               
                                                                                                  
 lstm_6 (LSTM)                  (None, 10, 128)      89088       ['onehot_input[0][0]']           
                                                                                                  
 den

59/59 [==============================] - 1s 23ms/step - loss: 0.4395 - accuracy: 0.0226 - val_loss: 0.4449 - val_accuracy: 0.0086 - lr: 0.0010
Epoch 21/100
59/59 [==============================] - 1s 22ms/step - loss: 0.4396 - accuracy: 0.0290 - val_loss: 0.4388 - val_accuracy: 0.0172 - lr: 0.0010
Epoch 22/100
59/59 [==============================] - 1s 23ms/step - loss: 0.4394 - accuracy: 0.0236 - val_loss: 0.4395 - val_accuracy: 0.0258 - lr: 0.0010
Epoch 23/100
59/59 [==============================] - 1s 22ms/step - loss: 0.4388 - accuracy: 0.0258 - val_loss: 0.4416 - val_accuracy: 0.0043 - lr: 0.0010
Epoch 24/100
59/59 [==============================] - 1s 22ms/step - loss: 0.4381 - accuracy: 0.0247 - val_loss: 0.4379 - val_accuracy: 0.0086 - lr: 0.0010
Epoch 25/100
59/59 [==============================] - 1s 23ms/step - loss: 0.4366 - accuracy: 0.0247 - val_loss: 0.4381 - val_accuracy: 0.0215 - lr: 0.0010
Epoch 26/100
59/59 [==============================] - 1s 22ms/step - loss: 0.

# css 적용

In [ ]:
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>🎲 AI 로또 예측기</title>
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            color: #333;
        }

        .container {
            max-width: 420px;
            margin: 0 auto;
            padding: 20px;
            min-height: 100vh;
        }

        .header {
            text-align: center;
            color: white;
            margin-bottom: 30px;
        }

        .header h1 {
            font-size: 2.5em;
            margin-bottom: 10px;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.3);
        }

        .header p {
            opacity: 0.9;
            font-size: 1.1em;
        }

        .card {
            background: rgba(255, 255, 255, 0.95);
            border-radius: 20px;
            padding: 25px;
            margin-bottom: 20px;
            box-shadow: 0 8px 32px rgba(0,0,0,0.1);
            backdrop-filter: blur(10px);
        }

        .status {
            text-align: center;
            padding: 15px;
            border-radius: 15px;
            margin-bottom: 20px;
            font-weight: bold;
        }

        .status.loading {
            background: linear-gradient(45deg, #f093fb 0%, #f5576c 100%);
            color: white;
        }

        .status.success {
            background: linear-gradient(45deg, #4facfe 0%, #00f2fe 100%);
            color: white;
        }

        .status.error {
            background: linear-gradient(45deg, #fa709a 0%, #fee140 100%);
            color: white;
        }

        .btn {
            width: 100%;
            padding: 18px;
            border: none;
            border-radius: 15px;
            font-size: 1.2em;
            font-weight: bold;
            cursor: pointer;
            transition: all 0.3s ease;
            margin-bottom: 15px;
        }

        .btn-primary {
            background: linear-gradient(45deg, #667eea 0%, #764ba2 100%);
            color: white;
        }

        .btn-secondary {
            background: linear-gradient(45deg, #f093fb 0%, #f5576c 100%);
            color: white;
        }

        .btn:hover {
            transform: translateY(-2px);
            box-shadow: 0 5px 15px rgba(0,0,0,0.2);
        }

        .btn:disabled {
            opacity: 0.6;
            cursor: not-allowed;
            transform: none;
        }

        .progress-bar {
            width: 100%;
            height: 8px;
            background: rgba(255,255,255,0.3);
            border-radius: 4px;
            overflow: hidden;
            margin: 15px 0;
        }

        .progress-fill {
            height: 100%;
            background: linear-gradient(45deg, #4facfe 0%, #00f2fe 100%);
            border-radius: 4px;
            transition: width 0.5s ease;
            width: 0%;
        }

        .lotto-numbers {
            display: flex;
            justify-content: center;
            align-items: center;
            flex-wrap: wrap;
            gap: 8px;
            margin: 15px 0;
        }

        .lotto-ball {
            width: 40px;
            height: 40px;
            border-radius: 50%;
            display: flex;
            align-items: center;
            justify-content: center;
            color: white;
            font-weight: bold;
            font-size: 14px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.2);
        }

        .ball-1to10 { background: linear-gradient(45deg, #ffa726, #ff7043); }
        .ball-11to20 { background: linear-gradient(45deg, #42a5f5, #1e88e5); }
        .ball-21to30 { background: linear-gradient(45deg, #ef5350, #e53935); }
        .ball-31to40 { background: linear-gradient(45deg, #66bb6a, #43a047); }
        .ball-41to45 { background: linear-gradient(45deg, #ab47bc, #8e24aa); }
        .ball-bonus { background: linear-gradient(45deg, #ffd54f, #ff8f00); }

        .prediction-item {
            background: linear-gradient(45deg, #f8f9fa, #e9ecef);
            border-radius: 15px;
            padding: 20px;
            margin-bottom: 15px;
            box-shadow: 0 4px 12px rgba(0,0,0,0.1);
        }

        .prediction-header {
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 15px;
        }

        .prediction-rank {
            background: linear-gradient(45deg, #667eea, #764ba2);
            color: white;
            width: 30px;
            height: 30px;
            border-radius: 50%;
            display: flex;
            align-items: center;
            justify-content: center;
            font-weight: bold;
        }

        .confidence {
            background: rgba(102, 126, 234, 0.1);
            color: #667eea;
            padding: 5px 12px;
            border-radius: 20px;
            font-size: 0.9em;
            font-weight: bold;
        }

        .data-info {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 15px;
            margin-bottom: 20px;
        }

        .info-item {
            text-align: center;
            padding: 15px;
            background: linear-gradient(45deg, #f8f9fa, #e9ecef);
            border-radius: 12px;
        }

        .info-value {
            font-size: 1.5em;
            font-weight: bold;
            color: #667eea;
        }

        .info-label {
            font-size: 0.9em;
            color: #666;
            margin-top: 5px;
        }

        .spinner {
            border: 3px solid rgba(255,255,255,0.3);
            border-radius: 50%;
            border-top: 3px solid white;
            width: 30px;
            height: 30px;
            animation: spin 1s linear infinite;
            margin: 0 auto;
        }

        @keyframes spin {
            0% { transform: rotate(0deg); }
            100% { transform: rotate(360deg); }
        }

        .hidden {
            display: none;
        }

        .fade-in {
            animation: fadeIn 0.5s ease-in;
        }

        @keyframes fadeIn {
            from { opacity: 0; transform: translateY(20px); }
            to { opacity: 1; transform: translateY(0); }
        }

        .footer {
            text-align: center;
            color: rgba(255,255,255,0.8);
            margin-top: 30px;
            font-size: 0.9em;
        }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1>🎲 AI 로또 예측기</h1>
            <p>딥러닝으로 분석하는 차세대 로또 번호 예측</p>
        </div>

        <!-- 데이터 정보 카드 -->
        <div class="card">
            <h3 style="margin-bottom: 15px;">📊 데이터 현황</h3>
            <div class="data-info">
                <div class="info-item">
                    <div class="info-value" id="totalDraws">-</div>
                    <div class="info-label">총 회차</div>
                </div>
                <div class="info-item">
                    <div class="info-value" id="latestDraw">-</div>
                    <div class="info-label">최신 회차</div>
                </div>
            </div>
            
            <button class="btn btn-secondary" onclick="updateData()" id="updateBtn">
                🔄 데이터 업데이트
            </button>
            
            <div id="updateStatus" class="status hidden">
                <div class="spinner"></div>
                <div style="margin-top: 10px;">데이터 업데이트 중...</div>
            </div>
        </div>

        <!-- 예측 실행 카드 -->
        <div class="card">
            <h3 style="margin-bottom: 15px;">🎯 번호 예측</h3>
            
            <button class="btn btn-primary" onclick="generatePredictions()" id="predictBtn">
                ✨ AI 번호 생성
            </button>
            
            <div id="predictionStatus" class="status hidden">
                <div class="spinner"></div>
                <div style="margin-top: 10px;">AI가 번호를 분석 중...</div>
            </div>
            
            <div class="progress-bar hidden" id="progressBar">
                <div class="progress-fill" id="progressFill"></div>
            </div>
        </div>

        <!-- 예측 결과 카드 -->
        <div class="card hidden" id="resultsCard">
            <h3 style="margin-bottom: 20px;">🏆 추천 번호 조합</h3>
            <div id="predictionResults"></div>
        </div>

        <div class="footer">
            <p>⚠️ 로또는 순수한 확률 게임입니다</p>
            <p>책임감 있는 복권 구매를 권장합니다</p>
        </div>
    </div>

    <script>
        // 가상의 로또 데이터 및 AI 모델 시뮬레이션
        let lottoData = [];
        let isModelTrained = false;

        // 초기 데이터 로드
        window.onload = function() {
            initializeData();
        };

        function initializeData() {
            // 실제로는 서버에서 CSV 데이터를 불러옴
            // 여기서는 시뮬레이션용 데이터 생성
            document.getElementById('totalDraws').textContent = '1175';
            document.getElementById('latestDraw').textContent = '1175';
        }

        async function updateData() {
            const updateBtn = document.getElementById('updateBtn');
            const updateStatus = document.getElementById('updateStatus');
            
            updateBtn.disabled = true;
            updateStatus.classList.remove('hidden');
            updateStatus.className = 'status loading';
            
            try {
                // 실제로는 웹크롤링 실행
                await simulateDataUpdate();
                
                updateStatus.className = 'status success';
                updateStatus.innerHTML = '✅ 데이터 업데이트 완료!';
                
                // 최신 회차 업데이트
                const currentDraw = parseInt(document.getElementById('latestDraw').textContent);
                document.getElementById('latestDraw').textContent = currentDraw + 1;
                
            } catch (error) {
                updateStatus.className = 'status error';
                updateStatus.innerHTML = '❌ 업데이트 실패: ' + error.message;
            }
            
            setTimeout(() => {
                updateStatus.classList.add('hidden');
                updateBtn.disabled = false;
            }, 3000);
        }

        async function generatePredictions() {
            const predictBtn = document.getElementById('predictBtn');
            const predictionStatus = document.getElementById('predictionStatus');
            const progressBar = document.getElementById('progressBar');
            const progressFill = document.getElementById('progressFill');
            const resultsCard = document.getElementById('resultsCard');
            
            predictBtn.disabled = true;
            predictionStatus.classList.remove('hidden');
            predictionStatus.className = 'status loading';
            progressBar.classList.remove('hidden');
            
            try {
                // 진행률 시뮬레이션
                for (let i = 0; i <= 100; i += 10) {
                    progressFill.style.width = i + '%';
                    await sleep(200);
                }
                
                // AI 모델 실행 시뮬레이션
                const predictions = await simulateAIPrediction();
                
                predictionStatus.className = 'status success';
                predictionStatus.innerHTML = '🎉 AI 분석 완료!';
                
                displayPredictions(predictions);
                resultsCard.classList.remove('hidden');
                resultsCard.classList.add('fade-in');
                
            } catch (error) {
                predictionStatus.className = 'status error';
                predictionStatus.innerHTML = '❌ 예측 실패: ' + error.message;
            }
            
            setTimeout(() => {
                predictionStatus.classList.add('hidden');
                progressBar.classList.add('hidden');
                predictBtn.disabled = false;
            }, 3000);
        }

        function displayPredictions(predictions) {
            const resultsContainer = document.getElementById('predictionResults');
            resultsContainer.innerHTML = '';
            
            predictions.forEach((pred, index) => {
                const predItem = document.createElement('div');
                predItem.className = 'prediction-item';
                
                predItem.innerHTML = `
                    <div class="prediction-header">
                        <div class="prediction-rank">${index + 1}</div>
                        <div class="confidence">신뢰도 ${(pred.confidence * 100).toFixed(1)}%</div>
                    </div>
                    <div class="lotto-numbers">
                        ${pred.main.map(num => `<div class="lotto-ball ${getBallClass(num)}">${num}</div>`).join('')}
                        <div style="margin: 0 10px; font-size: 1.2em; font-weight: bold;">+</div>
                        <div class="lotto-ball ball-bonus">${pred.bonus}</div>
                    </div>
                `;
                
                resultsContainer.appendChild(predItem);
            });
        }

        function getBallClass(num) {
            if (num <= 10) return 'ball-1to10';
            if (num <= 20) return 'ball-11to20';
            if (num <= 30) return 'ball-21to30';
            if (num <= 40) return 'ball-31to40';
            return 'ball-41to45';
        }

        // 시뮬레이션 함수들
        async function simulateDataUpdate() {
            // 실제로는 Python 웹크롤링 스크립트 실행
            await sleep(2000);
            return true;
        }

        async function simulateAIPrediction() {
            // 실제로는 Python 딥러닝 모델 실행
            await sleep(3000);
            
            // 현실적인 로또 번호 조합 생성
            const predictions = [];
            
            for (let i = 0; i < 10; i++) {
                const mainNumbers = generateRealisticNumbers();
                const bonusNumber = Math.floor(Math.random() * 45) + 1;
                
                // 메인 번호와 보너스 번호가 겹치지 않도록
                while (mainNumbers.includes(bonusNumber)) {
                    bonusNumber = Math.floor(Math.random() * 45) + 1;
                }
                
                predictions.push({
                    main: mainNumbers,
                    bonus: bonusNumber,
                    confidence: Math.random() * 0.3 + 0.4 // 0.4-0.7 사이
                });
            }
            
            // 신뢰도 순으로 정렬
            return predictions.sort((a, b) => b.confidence - a.confidence);
        }

        function generateRealisticNumbers() {
            const numbers = [];
            const zones = [
                { min: 1, max: 15, count: 2 },   // 1-15 구간에서 2개
                { min: 16, max: 30, count: 2 },  // 16-30 구간에서 2개  
                { min: 31, max: 45, count: 2 }   // 31-45 구간에서 2개
            ];
            
            zones.forEach(zone => {
                for (let i = 0; i < zone.count; i++) {
                    let num;
                    do {
                        num = Math.floor(Math.random() * (zone.max - zone.min + 1)) + zone.min;
                    } while (numbers.includes(num));
                    numbers.push(num);
                }
            });
            
            return numbers.sort((a, b) => a - b);
        }

        function sleep(ms) {
            return new Promise(resolve => setTimeout(resolve, ms));
        }

        // PWA 관련 기능
        if ('serviceWorker' in navigator) {
            window.addEventListener('load', function() {
                navigator.serviceWorker.register('/sw.js')
                    .then(function(registration) {
                        console.log('SW registered: ', registration);
                    })
                    .catch(function(registrationError) {
                        console.log('SW registration failed: ', registrationError);
                    });
            });
        }

        // 설치 프롬프트
        let deferredPrompt;
        window.addEventListener('beforeinstallprompt', (e) => {
            e.preventDefault();
            deferredPrompt = e;
            
            // 설치 버튼 표시 (필요시)
        });
    </script>
</body>
</html>

In [28]:
# 12) 정확도 계산 (전체 라벨 완전 일치율)
accuracy = np.mean(np.all(y_pred_bin == Y_val, axis=1))
print("\n🎯 전체 라벨 완전 일치 정확도 (Accuracy):", accuracy)


🎯 전체 라벨 완전 일치 정확도 (Accuracy): 0.0


## 3. 예측

In [22]:
pred = model.predict(X[-1].reshape(1,-1))[0]  # 마지막 데이터를 입력으로

# 0.5 이상인 숫자 추출 (예측된 번호)
predicted_numbers = [i+1 for i, p in enumerate(pred) if p > 0.5]

print("예측된 번호들:", predicted_numbers)


1/1 [==============================] - 0s 25ms/step
예측된 번호들: []
